In [548]:
import os
import argparse
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from datetime import datetime
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

In [549]:
import os
import logging
import argparse
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
sys.argv = ['']

In [550]:
event_csv = "patients_subset_10.csv" # 10, 100, all "Path to the main event CSV file."
event_path =  "./../../../commonfilesharePHI/slee/ckd-optum/" 

In [ ]:
# begin of tab 
df = pd.read_csv(event_path + event_csv, low_memory=False).drop_duplicates()
df.shape

In [552]:
df['EventTimeStamp'] = pd.to_datetime(df['EventTimeStamp'], errors='coerce')
df['EventDate'] = df['EventTimeStamp'].dt.date
df['DataCategory'] = df['DataCategory'].fillna('None')
df['DataNumeric'] = pd.to_numeric(df['DataNumeric'], errors='coerce')

In [553]:
df['is_gfr'] = df['DataCategory'].str.upper().str.contains("GFR|GFREST", na=False)
all_days = df[['PatientID', 'EventDate']].drop_duplicates().sort_values(['PatientID', 'EventDate'])

In [ ]:
df.shape

In [ ]:
df.head()

In [556]:
icd_filter = "\." # change, filter for ICD later
gfr_filter = "GFR|GFREST"

In [ ]:
icd_df = df[df['DataCategory'].str.upper().str.contains(icd_filter, na=False)]
icd_df.shape

In [ ]:
icd_df[['PatientID', 'EventDate']].drop_duplicates().shape

In [ ]:
gfr_df = df[df['DataCategory'].str.upper().str.contains(gfr_filter, na=False)]
gfr_df.shape

In [ ]:
gfr_df[['PatientID', 'EventDate']].drop_duplicates().shape

In [ ]:
# make sure no GFR after filtering for ICD in DataCategory
sum(df[df['DataCategory'].str.upper().str.contains(icd_filter, na=False)]['DataCategory'].str.upper().str.contains(gfr_filter, na=False))

In [ ]:
df.shape

In [ ]:
all_days.shape

In [ ]:
all_days.head()

In [ ]:
# change, discrepancy occurs here
print(df.shape)
gfr_df = df[df['is_gfr'] & df['DataNumeric'].notna()]
print(gfr_df.shape)
gfr_daywise = (
    gfr_df.groupby(['PatientID', 'EventDate'])['DataNumeric']
    .first().reset_index().rename(columns={'DataNumeric': 'GFR_combined'})
)

base_df = pd.merge(all_days, gfr_daywise, on=['PatientID', 'EventDate'], how='left')
print(base_df.shape)
base_df = base_df.sort_values(['PatientID', 'EventDate'])
base_df["GFR_combined"] = base_df.groupby("PatientID")["GFR_combined"].ffill()
print(base_df.shape)

In [ ]:
base_df.shape

In [ ]:
base_df

In [568]:
def gfr_to_stage(gfr):
    if pd.isna(gfr): 
        return None, 0
    if gfr >= 90: 
        return "1", 1
    if gfr >= 60: 
        return "2", 2
    if gfr >= 45: 
        return "3a", 3.1
    if gfr >= 30: 
        return "3b", 3.2
    if gfr >= 15: 
        return "4", 4
    return "5", 5

# Enforce monotonic CKD staging
new_stages = {}
for pid, group in base_df.groupby("PatientID"):  # tqdm can be re-enabled here
    group = group.sort_values("EventDate")
    max_rank = 0
    prev_idx = None
    for idx, row in group.iterrows():
        stage, rank = gfr_to_stage(row["GFR_combined"])
        if rank < max_rank:
            stage = new_stages.get(prev_idx, stage)
        else:
            max_rank = rank
        new_stages[idx] = stage
        prev_idx = idx

base_df["CKD_stage"] = base_df.index.map(new_stages)


In [ ]:
base_df.shape

In [570]:
# -----------------------------
# One-hot encode diagnoses (truncated ICD codes)
# -----------------------------
def truncate_icd(code):
    code = str(code).strip().replace(" ", "")
    if '.' in code:
        prefix, suffix = code.split('.', 1)
        return f"{prefix}.{suffix[0]}" if suffix else prefix
    return code


diag_df = df[df["DataType"] == "Diagnosis"].copy()
diag_df["ICD_clean"] = diag_df["DataCategory"].apply(truncate_icd)

In [ ]:
diag_df.shape

In [572]:
diagnosis_map = diag_df.groupby(["PatientID", "EventDate"])["ICD_clean"].apply(list)
mlb_diag = MultiLabelBinarizer()
diag_features = mlb_diag.fit_transform(diagnosis_map.values)

diag_df_onehot = pd.DataFrame(
    diag_features,
    columns=[f"diag_{c}" for c in mlb_diag.classes_],
    index=diagnosis_map.index
).reset_index()

base_df = pd.merge(base_df, diag_df_onehot, on=["PatientID", "EventDate"], how="left")

In [ ]:
base_df.shape

In [574]:
# -----------------------------
# One-hot encode medications
# -----------------------------
med_df = df[df["DataType"] == "Medications"].copy()
med_df["med_clean"] = med_df["DataCategory"].astype(str).str.upper().str.replace(" ", "_")

medication_map = med_df.groupby(["PatientID", "EventDate"])["med_clean"].apply(list)
mlb_med = MultiLabelBinarizer()
med_features = mlb_med.fit_transform(medication_map.values)

med_df_onehot = pd.DataFrame(
    med_features,
    columns=[f"med_{c}" for c in mlb_med.classes_],
    index=medication_map.index
).reset_index()

base_df = pd.merge(base_df, med_df_onehot, on=["PatientID", "EventDate"], how="left")


In [ ]:
base_df.shape

In [576]:

# -----------------------------
# Pivot-style lab expansion
# -----------------------------
lab_df = df[(df["DataType"] == "Labs") & df["DataNumeric"].notna()].copy()
lab_df["DataCategory"] = lab_df["DataCategory"].astype(str).str.upper()

lab_pivot = (
    lab_df.groupby(["PatientID", "EventDate", "DataCategory"])["DataNumeric"]
    .first().unstack("DataCategory").reset_index()
)

lab_pivot.columns = ["PatientID", "EventDate"] + [f"lab_{c}" for c in lab_pivot.columns[2:]]
base_df = pd.merge(base_df, lab_pivot, on=["PatientID", "EventDate"], how="left")


In [ ]:
base_df.shape

In [578]:
# -----------------------------
# Optional: One-hot encode demographics
# -----------------------------
def format_demographics(row):
    race_ethnicity = str(row["DataCategory"]).replace("//", " ").replace("/", " ")
    if "Unknown Not Reported" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Unknown Not Reported", "").strip()
    if "Do not identify with Race" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Do not identify with Race", "unknown race").strip()
    return race_ethnicity

def build_demographic_map(df):
    demo_df = df[df["DataType"] == "Demographics"].dropna(subset=["DataCategory"])
    return demo_df.groupby("PatientID").first().apply(format_demographics, axis=1).to_dict()

demo_map = build_demographic_map(df)
demo_df = pd.DataFrame(list(demo_map.items()), columns=["PatientID", "demo_string"])


In [ ]:
demo_df.shape

In [580]:
if not demo_df.empty:
    enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    demo_encoded = enc.fit_transform(demo_df[["demo_string"]])
    demo_onehot = pd.DataFrame(demo_encoded, columns=[f"demo_{c}" for c in enc.categories_[0]])
    demo_df = pd.concat([demo_df[["PatientID"]], demo_onehot], axis=1)
else:
    demo_df = pd.DataFrame(columns=["PatientID"])

base_df = pd.merge(base_df, demo_df, on="PatientID", how="left")

In [ ]:
base_df.shape

In [582]:
tab_shape = base_df.shape[0]

In [583]:
# last tab

In [584]:
# begin of embedding

In [585]:
def parse_arguments():
    parser = argparse.ArgumentParser(
        description="Generate synthetic patient-day notes, map GFR to CKD stages, and generate embeddings using a transformer model."
    )
    parser.add_argument("--csv", type=str, default=event_path + event_csv,
                        help="Path to the main event CSV file.")
    parser.add_argument("--icd", type=str, default=event_path + "icd_mapping.csv",
                        help="Path to the ICD mapping CSV file.")
                        # change
    parser.add_argument("--output_dir", type=str, default=event_path +embedding_gen_dir,
                        help="Directory in which to save the generated embeddings and metadata.")
    parser.add_argument("--model_name", type=str, default="./../../../commonfilesharePHI/slee/GeneratEHR/clinicalBERT-emily",
                        help="Pretrained transformer model to use for embeddings.")
    parser.add_argument("--embed_dim", type=int, default=768,
                        help="Dimension to which the model embedding should be truncated or padded.")
    parser.add_argument("--batch_size", type=int, default=128,
                        help="Batch size for encoding the synthetic notes.")
    return parser.parse_args()

def load_data(csv_path, icd_path):
    print(f"[INFO] Loading patient events from: {csv_path}")
    # Set low_memory to False to suppress dtype warnings for mixed types.
    df = pd.read_csv(csv_path, low_memory=False)
    df = df.drop_duplicates()
    icd_df = pd.read_csv(icd_path)

    df['DataCategory'] = df['DataCategory'].fillna('None')
    df['DataNumeric'] = df['DataNumeric'].fillna('None')
    df['EventTimeStamp'] = pd.to_datetime(df['EventTimeStamp'], errors='coerce')
    df['EventDate'] = df['EventTimeStamp'].dt.date
    df['is_gfr'] = df['DataCategory'].str.upper().str.contains('GFR|GFREST', na=False)

    icd_df["icd_code"] = icd_df["icd_code"].astype(str).str.replace(".", "", regex=False)
    icd_map = dict(zip(icd_df["icd_code"], icd_df["long_title"]))
    return df, icd_map

In [586]:
args = parse_arguments()

In [ ]:
df, icd_map = load_data(args.csv, args.icd)

In [ ]:
df.shape

In [ ]:
df.head()

In [590]:

def format_demographics(row):
    # When grouping by PatientID without resetting index, PatientID is in row.name.
    pid = row.name
    race_ethnicity = str(row["DataCategory"]).replace("//", " ").replace("/", " ")
    if "Unknown Not Reported" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Unknown Not Reported", "").strip()
    if "Do not identify with Race" in race_ethnicity:
        race_ethnicity = race_ethnicity.replace("Do not identify with Race", "unknown race").strip()
    return f"Patient {pid} is a {race_ethnicity} patient."
    
def build_demographic_map(df):
    demographics = df[df["DataType"] == "Demographics"].dropna(subset=["DataCategory"])
    demographic_map = (
        demographics.groupby("PatientID")
        .first()
        .apply(format_demographics, axis=1)
        .to_dict()
    )
    return demographic_map

In [591]:
demographic_map = build_demographic_map(df)

In [592]:
# change, discrepancy occurs here
def generate_synthetic_notes(df, demographic_map, icd_map):
    print(df.shape)
    # events vs df
    events = df.copy()
    # events = df[df["DataType"] != "Demographics"].copy() # original
    print(events.shape)
    print("difference", df.shape[0] - events.shape[0])
    grouped = events.groupby(['PatientID', 'EventDate'])
    print(grouped.size())
    records = []

    for (pid, date), group in tqdm(grouped, desc="Formatting synthetic notes"):
        note_lines = []
        gfr = None

        if pid in demographic_map:
            note_lines.append(demographic_map[pid])
        else:
            note_lines.append(f"Patient {pid} demographics information not available.")

        date_str = datetime.strftime(pd.Timestamp(date), "%Y-%m-%d")
        note_lines.append(f"On {date_str}, the patient had the following records:")

        for _, row in group.iterrows():
            dt, cat, num = row['DataType'], row['DataCategory'], row['DataNumeric']
            if dt == "Diagnosis":
                icd_code = str(cat).replace(".", "")
                icd_title = icd_map.get(icd_code, "Unknown condition")
                note_lines.append(f"  - ICD-10 code {cat}: {icd_title}")
            elif dt == "Medication":
                note_lines.append(f"  - Medication administered: {cat}")
            elif dt == "Procedure":
                note_lines.append(f"  - Procedure performed: {cat}")
            else:
                note_lines.append(f"  - {dt}: {cat}")

            if row['is_gfr']:
                try:
                    gfr_candidate = float(num)
                    if gfr is None:
                        gfr = gfr_candidate
                except Exception:
                    continue

        full_note = "\n".join(note_lines)
        records.append({'PatientID': pid, 'EventDate': date, 'text': full_note, 'GFR': gfr})
    print(len(records))
    summary_df = pd.DataFrame(records)
    print(f"[INFO] Generated {len(summary_df)} synthetic patient-day notes.")
    return summary_df


In [ ]:
summary_df = generate_synthetic_notes(df, demographic_map, icd_map)

In [ ]:
summary_df

In [595]:
def gfr_to_stage(gfr):
    if pd.isna(gfr):
        return None, 0
    if gfr >= 90:
        return "1", 1
    elif gfr >= 60:
        return "2", 2
    elif gfr >= 45:
        return "3a", 3.1
    elif gfr >= 30:
        return "3b", 3.2
    elif gfr >= 15:
        return "4", 4
    else:
        return "5", 5

def forward_fill_ckd_stage(summary_df):
    """
    For each patient, forward-fill the GFR values (sorted by date), and map them to CKD stages
    based on the following thresholds:
    
        Stage 1: eGFR ≥ 90
        Stage 2: 60 ≤ eGFR < 90
        Stage 3a: 45 ≤ eGFR < 60
        Stage 3b: 30 ≤ eGFR < 45
        Stage 4: 15 ≤ eGFR < 30
        Stage 5: eGFR < 15
    
    The stage is forced to be non-decreasing (i.e. if a new reading would lead to an improvement,
    the previous worse stage is retained).
    """
    summary_df = summary_df.sort_values(by=["PatientID", "EventDate"]).copy()
    # Convert GFR to numeric (if not already) and forward fill per patient.
    summary_df["GFR"] = pd.to_numeric(summary_df["GFR"], errors="coerce")
    summary_df["GFR"] = summary_df.groupby("PatientID")["GFR"].ffill()
    
    # For each patient, enforce non-decreasing (progressive) stage.
    new_stages = {}
    for pid, group in summary_df.groupby("PatientID"):
        group = group.sort_values("EventDate")
        max_stage_rank = 0
        for idx, row in group.iterrows():
            computed_stage, rank = gfr_to_stage(row["GFR"])
            # If the computed stage is less severe than the worst seen so far, retain the worst.
            if rank < max_stage_rank:
                final_stage = new_stages.get(prev_idx, computed_stage)
            else:
                final_stage = computed_stage
                max_stage_rank = rank
            new_stages[idx] = final_stage
            prev_idx = idx
    summary_df["CKD_stage"] = summary_df.index.map(new_stages)
    return summary_df

In [ ]:
# Forward-fill GFR values and compute CKD stage per patient
summary_df = forward_fill_ckd_stage(summary_df)
print(summary_df.head())

In [ ]:
summary_df.shape

In [ ]:
print("difference")
tab_shape - summary_df.shape[0]

In [599]:

def load_embedding_model(model_name, device):
    print(f"[INFO] Loading model from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tokenizer, model

def get_cls_embeddings(texts, tokenizer, model, device, embed_dim):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    cls_emb = outputs.last_hidden_state[:, 0, :]
    if cls_emb.size(1) > embed_dim:
        cls_emb = cls_emb[:, :embed_dim]
    else:
        pad = embed_dim - cls_emb.size(1)
        cls_emb = torch.nn.functional.pad(cls_emb, (0, pad), value=0)
    return cls_emb.cpu().numpy()

def generate_and_save_embeddings(summary_df, tokenizer, model, device, embed_dim, batch_size, output_dir):
    meta = []
    texts = summary_df['text'].tolist()
    ids = list(zip(summary_df['PatientID'], summary_df['EventDate']))
    gfrs = summary_df['GFR'].tolist()

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding notes in batches"):
        batch_texts = texts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        batch_gfrs = gfrs[i:i+batch_size]

        emb = get_cls_embeddings(batch_texts, tokenizer, model, device, embed_dim)

        for (pid, date), gfr_val, vec in zip(batch_ids, batch_gfrs, emb):
            # Create a folder for the patient if it doesn't exist.
            patient_folder = os.path.join(output_dir, str(pid))
            os.makedirs(patient_folder, exist_ok=True)

            date_str = pd.to_datetime(date).strftime('%Y%m%d')
            fname = f"{pid}_{date_str}.npz"
            fpath = os.path.join(patient_folder, fname)
            np.savez_compressed(fpath, cls_embedding=vec)
            # Look up the CKD stage from the summary dataframe.
            stage_val = summary_df[(summary_df['PatientID'] == pid) & (summary_df['EventDate'] == date)]['CKD_stage'].values[0]
            meta.append({
                'PatientID': pid,
                'EventDate': date,
                'GFR': gfr_val,
                'CKD_stage': stage_val,
                'text': summary_df[(summary_df['PatientID'] == pid) & (summary_df['EventDate'] == date)]['text'].values[0],
                'embedding_file': os.path.join(str(pid), fname)
            })

    meta_df = pd.DataFrame(meta)
    print(meta_df.shape)
    meta_csv_path = os.path.join(output_dir, 'patient_embedding_metadata.csv')
    # meta_df.to_csv(meta_csv_path, index=False)
    print(f"[DONE] Metadata saved to: {meta_csv_path}")

In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
tokenizer, model = load_embedding_model(args.model_name, device)
generate_and_save_embeddings(summary_df, tokenizer, model, device,
                                args.embed_dim, args.batch_size, args.output_dir)


In [264]:
# last of embedding

In [ ]:
# verify gen with output

In [ ]:
meta_df = pd.read_csv(event_path +  "ckd_embeddings_100/" + 'patient_embedding_metadata.csv')


In [ ]:
meta_df.shape

In [440]:
# test new ouput dir

In [ ]:
os.listdir("./../../../commonfilesharePHI/ldiao/ckd_project/")